# Diabetes EDL Clinical Decision Support System — Full Keras Pipeline

Replicates the Ensemble Deep Learning (EDL) methodology of:

> M. S. Al Reshan et al., *"An Innovative Ensemble Deep Learning Clinical Decision Support System for Diabetes Prediction,"* IEEE Access, vol. 12, 2024.

using **real Keras ANN, LSTM and CNN base models**, combined with **real Stack-ANN, Stack-LSTM and Stack-CNN meta-level models**, on two real-world datasets:

- **BRFSS 2015 Diabetes Health Indicators** (253,680 rows, 3-class)
- **Diabetic Patient Readmission** (101,766 rows, binary: readmitted <30 days)

**How to use this notebook:**
1. Runtime → Change runtime type → GPU (optional but faster)
2. Run the "Setup" cell to install packages
3. Run the "Upload data" cell and pick your two CSV files when prompted
4. Run all remaining cells top to bottom


## 1. Setup — install packages

In [ ]:
!pip install -q imbalanced-learn scikit-learn tensorflow pandas numpy matplotlib
print("Setup complete.")


## 2. Upload your data

Upload `diabetes_012_health_indicators_BRFSS2015.csv` and `diabetic_data.csv`.
(If you'd rather use Google Drive, comment this cell out and mount Drive instead with
`from google.colab import drive; drive.mount('/content/drive')`, then set `DATA_DIR` below.)


In [ ]:
from google.colab import files
uploaded = files.upload()  # select both CSVs when prompted
DATA_DIR = "."


## 3. Imports and reproducibility

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Flatten, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, matthews_corrcoef, roc_auc_score,
                              confusion_matrix)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


## 4. Evaluation metrics (mirrors the paper's Equations 1-6)

In [ ]:
def specificity_score(y_true, y_pred, n_classes):
    labels = list(range(n_classes))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    total = cm.sum()
    specs = []
    for i in labels:
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(specs))


def evaluate(y_true, y_pred, y_proba, n_classes):
    binary = n_classes == 2
    avg = "binary" if binary else "macro"
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, average=avg, zero_division=0),
        "specificity": specificity_score(y_true, y_pred, n_classes),
        "f_score": f1_score(y_true, y_pred, average=avg, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    try:
        if binary:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba[:, 1])
        else:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except Exception:
        metrics["roc_auc"] = float("nan")
    return {k: round(float(v), 4) for k, v in metrics.items()}


## 5. Data loading

In [ ]:
def load_brfss(data_dir):
    df = pd.read_csv(f"{data_dir}/diabetes_012_health_indicators_BRFSS2015.csv").drop_duplicates()
    y = df["Diabetes_012"].astype(int).values
    X = df.drop(columns=["Diabetes_012"])
    return X, y, 3, ["No Diabetes", "Prediabetes", "Diabetes"]


def load_readmit(data_dir):
    df = pd.read_csv(f"{data_dir}/diabetic_data.csv").replace("?", np.nan)
    df = df.drop(columns=["encounter_id", "patient_nbr", "weight", "payer_code", "medical_specialty"])
    df = df.drop_duplicates()
    y = (df["readmitted"] == "<30").astype(int).values
    X = df.drop(columns=["readmitted"])
    for col in X.columns:
        if X[col].dtype == object:
            X[col] = X[col].astype(str)
            X[col] = LabelEncoder().fit_transform(X[col])
    X = X.apply(pd.to_numeric, errors="coerce")
    return X, y, 2, ["Not Readmitted <30d", "Readmitted <30d"]


## 6. Preprocessing: clean, scale, Extra Tree Classifier feature selection, 80:20 split, SMOTE

In [ ]:
def preprocess(X, y, n_classes, top_k_features=15):
    X = X.fillna(X.mean(numeric_only=True))
    scaler = MinMaxScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

    etc = ExtraTreesClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
    etc.fit(X_scaled, y)
    importances = pd.Series(etc.feature_importances_, index=X_scaled.columns).sort_values(ascending=False)
    print("Top features (ETC importance):")
    print(importances.head(10))
    top_features = importances.head(top_k_features).index.tolist()
    X_selected = X_scaled[top_features].values

    X_train, X_test, y_train, y_test = train_test_split(
        X_selected, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    sm = SMOTE(random_state=RANDOM_STATE)
    X_train, y_train = sm.fit_resample(X_train, y_train)

    print(f"Train set after SMOTE: {X_train.shape}, Test set: {X_test.shape}")
    return X_train, X_test, y_train, y_test, top_features, importances


## 7. Base-level models — ANN, LSTM, CNN (Section IV-E-1 of the paper)

In [ ]:
def build_ann(input_dim, n_classes):
    out_units, out_act, loss = (1, "sigmoid", "binary_crossentropy") if n_classes == 2 \
        else (n_classes, "softmax", "categorical_crossentropy")
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dropout(0.3),
        Dense(out_units, activation=out_act),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss=loss, metrics=["accuracy"])
    return model


def build_lstm(input_dim, n_classes):
    out_units, out_act, loss = (1, "sigmoid", "binary_crossentropy") if n_classes == 2 \
        else (n_classes, "softmax", "categorical_crossentropy")
    model = Sequential([
        Input(shape=(input_dim, 1)),
        LSTM(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(out_units, activation=out_act),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss=loss, metrics=["accuracy"])
    return model


def build_cnn(input_dim, n_classes):
    out_units, out_act, loss = (1, "sigmoid", "binary_crossentropy") if n_classes == 2 \
        else (n_classes, "softmax", "categorical_crossentropy")
    model = Sequential([
        Input(shape=(input_dim, 1)),
        Conv1D(32, kernel_size=3, activation="relu", padding="same"),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation="relu"),
        Dense(out_units, activation=out_act),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss=loss, metrics=["accuracy"])
    return model


## 8. Meta-level stacking models — Stack-ANN, Stack-LSTM, Stack-CNN (paper's exact hyperparameters: LR 0.01, 20 epochs, batch 32)

In [ ]:
def build_stack_ann(input_dim, n_classes):
    out_units, out_act, loss = (1, "sigmoid", "binary_crossentropy") if n_classes == 2 \
        else (n_classes, "softmax", "categorical_crossentropy")
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, activation="relu"),
        Dropout(0.5),
        Dense(out_units, activation=out_act),
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
    return model


def build_stack_lstm(input_dim, n_classes):
    out_units, out_act, loss = (1, "sigmoid", "binary_crossentropy") if n_classes == 2 \
        else (n_classes, "softmax", "categorical_crossentropy")
    model = Sequential([
        Input(shape=(input_dim, 1)),
        LSTM(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(out_units, activation=out_act),
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
    return model


def build_stack_cnn(input_dim, n_classes):
    out_units, out_act, loss = (1, "sigmoid", "binary_crossentropy") if n_classes == 2 \
        else (n_classes, "softmax", "categorical_crossentropy")
    model = Sequential([
        Input(shape=(input_dim, 1)),
        Conv1D(32, kernel_size=3, activation="relu", padding="same"),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation="relu"),
        Dense(out_units, activation=out_act),
    ])
    model.compile(optimizer=Adam(learning_rate=0.01), loss=loss, metrics=["accuracy"])
    return model


def prep_target(y, n_classes):
    return y.astype("float32") if n_classes == 2 else to_categorical(y, num_classes=n_classes)


def get_proba(model, X, n_classes):
    p = model.predict(X, verbose=0)
    if n_classes == 2:
        p = p.reshape(-1)
        return np.column_stack([1 - p, p])
    return p


## 9. Full pipeline runner for one dataset

In [ ]:
def run(dataset, data_dir=DATA_DIR, epochs_base=20, epochs_stack=20, batch_size=32, top_k_features=15):
    if dataset == "brfss":
        X, y, n_classes, class_names = load_brfss(data_dir)
    elif dataset == "readmit":
        X, y, n_classes, class_names = load_readmit(data_dir)
    else:
        raise ValueError("dataset must be 'brfss' or 'readmit'")

    X_train, X_test, y_train, y_test, top_features, importances = preprocess(
        X, y, n_classes, top_k_features=top_k_features
    )
    input_dim = X_train.shape[1]
    y_train_enc = prep_target(y_train, n_classes)
    X_train_seq = X_train.reshape(-1, input_dim, 1)
    X_test_seq = X_test.reshape(-1, input_dim, 1)

    results = {}

    print("\n=== Training base ANN ===")
    ann = build_ann(input_dim, n_classes)
    ann.fit(X_train, y_train_enc, epochs=epochs_base, batch_size=batch_size, validation_split=0.2, verbose=1)
    ann_train_proba = get_proba(ann, X_train, n_classes)
    ann_test_proba = get_proba(ann, X_test, n_classes)
    ann_test_pred = np.argmax(ann_test_proba, axis=1) if n_classes > 2 else (ann_test_proba[:, 1] > 0.5).astype(int)
    results["ANN"] = evaluate(y_test, ann_test_pred, ann_test_proba, n_classes)

    print("\n=== Training base LSTM ===")
    lstm = build_lstm(input_dim, n_classes)
    lstm.fit(X_train_seq, y_train_enc, epochs=epochs_base, batch_size=batch_size, validation_split=0.2, verbose=1)
    lstm_train_proba = get_proba(lstm, X_train_seq, n_classes)
    lstm_test_proba = get_proba(lstm, X_test_seq, n_classes)
    lstm_test_pred = np.argmax(lstm_test_proba, axis=1) if n_classes > 2 else (lstm_test_proba[:, 1] > 0.5).astype(int)
    results["LSTM"] = evaluate(y_test, lstm_test_pred, lstm_test_proba, n_classes)

    print("\n=== Training base CNN ===")
    cnn = build_cnn(input_dim, n_classes)
    cnn.fit(X_train_seq, y_train_enc, epochs=epochs_base, batch_size=batch_size, validation_split=0.2, verbose=1)
    cnn_train_proba = get_proba(cnn, X_train_seq, n_classes)
    cnn_test_proba = get_proba(cnn, X_test_seq, n_classes)
    cnn_test_pred = np.argmax(cnn_test_proba, axis=1) if n_classes > 2 else (cnn_test_proba[:, 1] > 0.5).astype(int)
    results["CNN"] = evaluate(y_test, cnn_test_pred, cnn_test_proba, n_classes)

    meta_X_train = np.hstack([ann_train_proba, lstm_train_proba, cnn_train_proba])
    meta_X_test = np.hstack([ann_test_proba, lstm_test_proba, cnn_test_proba])
    meta_dim = meta_X_train.shape[1]
    meta_seq_train = meta_X_train.reshape(-1, meta_dim, 1)
    meta_seq_test = meta_X_test.reshape(-1, meta_dim, 1)

    print("\n=== Training Stack-ANN (meta) ===")
    stack_ann = build_stack_ann(meta_dim, n_classes)
    stack_ann.fit(meta_X_train, y_train_enc, epochs=epochs_stack, batch_size=batch_size, validation_split=0.2, verbose=1)
    sa_proba = get_proba(stack_ann, meta_X_test, n_classes)
    sa_pred = np.argmax(sa_proba, axis=1) if n_classes > 2 else (sa_proba[:, 1] > 0.5).astype(int)
    results["Stack-ANN"] = evaluate(y_test, sa_pred, sa_proba, n_classes)

    print("\n=== Training Stack-LSTM (meta) ===")
    stack_lstm = build_stack_lstm(meta_dim, n_classes)
    stack_lstm.fit(meta_seq_train, y_train_enc, epochs=epochs_stack, batch_size=batch_size, validation_split=0.2, verbose=1)
    sl_proba = get_proba(stack_lstm, meta_seq_test, n_classes)
    sl_pred = np.argmax(sl_proba, axis=1) if n_classes > 2 else (sl_proba[:, 1] > 0.5).astype(int)
    results["Stack-LSTM"] = evaluate(y_test, sl_pred, sl_proba, n_classes)

    print("\n=== Training Stack-CNN (meta) ===")
    stack_cnn = build_stack_cnn(meta_dim, n_classes)
    stack_cnn.fit(meta_seq_train, y_train_enc, epochs=epochs_stack, batch_size=batch_size, validation_split=0.2, verbose=1)
    sc_proba = get_proba(stack_cnn, meta_seq_test, n_classes)
    sc_pred = np.argmax(sc_proba, axis=1) if n_classes > 2 else (sc_proba[:, 1] > 0.5).astype(int)
    results["Stack-CNN"] = evaluate(y_test, sc_pred, sc_proba, n_classes)

    print("\n================ FINAL RESULTS ================")
    results_df = pd.DataFrame(results).T
    display(results_df)

    models = {"ann": ann, "lstm": lstm, "cnn": cnn,
              "stack_ann": stack_ann, "stack_lstm": stack_lstm, "stack_cnn": stack_cnn}
    for name, m in models.items():
        m.save(f"{dataset}_{name}.keras")

    with open(f"{dataset}_keras_results.json", "w") as f:
        json.dump({"dataset": dataset, "class_names": class_names,
                    "top_features": top_features, "results": results}, f, indent=2)

    return results_df, models


## 10. Run on Dataset A — BRFSS 2015 Diabetes Health Indicators

In [ ]:
results_brfss, models_brfss = run("brfss", epochs_base=20, epochs_stack=20, top_k_features=15)
results_brfss


## 11. Run on Dataset B — Diabetic Patient Readmission

In [ ]:
results_readmit, models_readmit = run("readmit", epochs_base=20, epochs_stack=20, top_k_features=20)
results_readmit


## 12. Compare accuracy across models (chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results_brfss["accuracy"].plot(kind="bar", ax=axes[0], title="BRFSS — Accuracy by model", color="steelblue")
results_readmit["accuracy"].plot(kind="bar", ax=axes[1], title="Readmission — Accuracy by model", color="darkorange")
for ax in axes:
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 13. Download your trained models and results

Each `run()` call saved 6 `.keras` model files and one `*_keras_results.json` per dataset
to the Colab file system. Download them (or save to Drive) before the runtime disconnects:


In [ ]:
from google.colab import files
import glob
for fpath in glob.glob("*.keras") + glob.glob("*_keras_results.json"):
    files.download(fpath)
